<!-- notebook-header -->
<a id="top"></a>

# 2. Arrays, Hashing & Prefix Sums

*LeetCode Playbook · notebook 2 of 12* · Remember what you have seen, and turn running totals into O(1) range questions.

**In this notebook:** [Arrays & Hashing](#s03) · [Prefix Sums](#s04)

← [Start Here](01_Start_Here.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Two Pointers, Sliding Window & Strings](03_Two_Pointers_Sliding_Window_Strings.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s03"></a>

## Arrays & Hashing

> A hash map is a notebook with instant recall. Walk the array once; at each item, first ask the notebook the one question the brute force would answer with a rescan ("is my partner in here?", "how many of these so far?", "which group is this?"), then write the item down. Each question costs O(1) on average, so the whole walk is O(n).

**Reach for it when** the brute force has an inner loop that only *searches* for a value (a partner, an earlier copy, a matching group); the problem says pairs, duplicates, frequency, anagrams, "group by", or wants O(n) on unsorted data; or the brute force compares items only by value (equal, sums to, same letters), never by where they sit.

**In this repo:** `arrays_hashing/` (17 problems; Subarray Sum Equals K, Product of Array Except Self and Max Consecutive Ones are taught in [Prefix Sums](#s04)) · bank: `practice/simple/01_two_sum.py`, `practice/simple/02_group_anagrams.py`, `practice/simple/04_longest_consecutive_sequence.py` · basics: `practice/simple/basics/strings/01_character_counting_and_anagrams.py`, `practice/simple/basics/strings/05_encode_decode_strings_and_join.py`, `practice/simple/basics/sorting/05_counting_and_bucket_sort.py` · the same tools elsewhere: `strings/valid_anagram.py`, `strings/group_shifted_strings.py`, `design/insert_delete_getrandom_o1.py`

### The picture

```text
nums = 3  5  2  7  11      target = 9           ask the notebook first, then write yourself in

i=0   [3] 5  2  7  11      need 6   notebook {}                no  -> write 3:0
i=1    3 [5] 2  7  11      need 4   notebook {3:0}             no  -> write 5:1
i=2    3  5 [2] 7  11      need 7   notebook {3:0, 5:1}        no  -> write 2:2
i=3    3  5  2 [7] 11      need 2   notebook {3:0, 5:1, 2:2}   YES -> answer [2, 3]
       '-----'
       everything left of the cursor is already in the notebook; 11 is never read
```

Why it is fast: once the brute force fixes `v`, it rescans the other numbers to find `target - v`, a value it *already knows*. A search by value is exactly what a hash map does in O(1) on average: it turns the value into a slot number (the hash) and jumps there. n items × O(1) = O(n) time, paid for with O(n) memory.

**Why it is correct:** every pair (i, j) with i < j is found exactly when j arrives: i is already in the notebook, and asking before writing means j never finds itself.

The notebook does not always hold positions. To group items, its key is a **fingerprint**: anything that is equal for items that belong together and different otherwise. For anagrams it is the letter counts:

```text
word   fingerprint            bucket
eat    a:1 e:1 t:1     --->   [eat, tea, ate]
tea    a:1 e:1 t:1     --->   (same bucket)
tan    a:1 n:1 t:1     --->   [tan, nat]
```

### From idea to code

**The idea in one sentence:** *walk once; for the current item, first ask the map the question the brute force would rescan for, then store what a later item will need to ask about.*

| Decision | Hashing answer |
|---|---|
| **State**: what must I remember? | one dict or set keyed by **what you will look up later**: value → index (pairs), value → count (frequency), fingerprint → list (groups) |
| **Definition**: what exactly does each variable mean? | write it as a comment: `seen[v] = index of an earlier v (left of i)` |
| **Invariant**: what is true at the end of every step? | *ask, then store*: at the top of step `i` the map describes exactly `nums[0..i-1]`. *Build first*: it describes the whole input before the scan starts |
| **Step**: how does one item change the state? | store the item: `seen[v] = i`, `count[v] += 1`, `groups[key].append(w)` |
| **Record**: when is the answer updated? | the moment a lookup succeeds: return the pair, add to a count, report a repeat |
| **Init**: starting values | an empty map (seeded when the empty prefix counts, like `{0: 1}` in [Prefix Sums](#s04)) |
| **Return**: what comes back, and for "not found"? | the pair, the count or `list(groups.values())`; `[]`, `-1` or `False` when nothing matched |

**From the question to the code.** Two decisions settle every hashing solution: the **key** (what a later lookup asks about) and **when you write**. *Ask, then store* when an item pairs with an **earlier** item. *Build first* when the question is about the whole collection.

| Question | Key → value | When to write | Record |
|---|---|---|---|
| a pair summing to target (1) | value → index | ask, then store | return at the first hit |
| how many pairs sum to a multiple of 60 (1010) | `t % 60` → count | ask, then store | `pairs += seen[(60 - r) % 60]` |
| a copy within distance k (219) | value → **last** index | ask, then overwrite | hit when `i - last[v] <= k` |
| same letters (49) | fingerprint → members | store only | `list(groups.values())` |
| k most frequent (347) | value → count | build first | buckets by count, read from the top |
| longest run of consecutive values (128) | value (a set) | build first | walk up from run starts |
| no repeat in a row, column or box (36) | `(unit, digit)` (a set) | ask, then store | `False` at the first repeat |
| `a + b + c + d == 0` over four lists (454) | `a + b` → count | build one half, ask with the other | `+= sums[-(c + d)]` |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "remember what I have seen, and where" | `seen = {}` |
| "the partner I need" | `need = target - v` |
| "have I seen it, and where?" | `if need in seen:` then `seen[need]` |
| "now remember me" | `seen[v] = i` (after the check) |
| "how many of each" | `count = Counter(nums)` or `count[v] += 1` |
| "put w in its group" | `groups[key].append(w)` with `groups = defaultdict(list)` |
| "a key made of a list" | `tuple(counts)` (a list can't be a key, a tuple can) |

Two templates cover the two shapes. In Two Sum the RECORD line comes *before* the STEP, because the lookup may only see earlier items ([From Idea to Code](01_Start_Here.ipynb#s01) shows what the swapped order breaks). In Group Anagrams storing the word *is* the answer, so STEP and RECORD are one line.

In [ ]:
def two_sum(nums, target):
    seen = {}                                # STATE + INIT: seen[v] = index of an earlier v (left of i)
    for i, v in enumerate(nums):
        need = target - v                    # the one partner that completes the sum
        if need in seen:                     # RECORD: ask the past first ...
            return [seen[need], i]
        seen[v] = i                          # STEP: ... then join it, so v never pairs with itself
    return []                                # RETURN: no pair


def group_anagrams(words):
    groups = defaultdict(list)               # STATE + INIT: fingerprint -> words with that fingerprint
    for w in words:
        counts = [0] * 26                    # the fingerprint: how many a's, b's, ..., z's (lowercase a-z only)
        for ch in w:
            counts[ord(ch) - ord("a")] += 1
        groups[tuple(counts)].append(w)      # STEP + RECORD: same letters, same key (a tuple can be a key)
    return list(groups.values())             # RETURN


print(two_sum([3, 5, 2, 7, 11], 9))          # [2, 3]
print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
# [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

**Try it**
- Use `frozenset(w)` as the key and run `group_anagrams(["aab", "abb"])`: `[['aab', 'abb']]`. A set forgets how many copies of each letter there are, so it is not a fingerprint.
- Replace `tuple(counts)` with `counts` and run `group_anagrams(["ab"])`: `TypeError: unhashable type: 'list'`. Dict keys must be hashable, and a list is not (it can change).
- Run `group_anagrams(["Zt", "tt"])`: `[['Zt', 'tt']]`. `ord("Z") - ord("a")` is −7, and `counts[-7]` is the slot of `t`. The fingerprint silently assumes lowercase a-z.

### Watch it work

Each line is one step; `seen` is shown *before* the current item is stored. Watch the first row: the 3 asks for a 3 while the notebook is still empty, so it cannot pair with itself.

In [ ]:
def trace_two_sum(nums, target):
    seen = {}
    for i, v in enumerate(nums):
        need = target - v
        found = need in seen
        action = f"found at index {seen[need]}" if found else f"store {v}: {i}"
        print(f"i={i}  v={v:<3} need={need:<3} seen={str(seen):<24} {action}")
        if found:
            return [seen[need], i]
        seen[v] = i
    return []


print(trace_two_sum([3, 2, 4], 6))   # [1, 2]

**Try it**
- Run `trace_two_sum([1, 1, 1, 3], 4)`: the last row shows `seen={1: 2}`. Each copy of 1 overwrote the previous one; any copy is a fine partner, and the answer is `[2, 3]`.
- Run `trace_two_sum([1, 2, 3], 100)`: every row stores, the result is `[]`, and `seen` ends up holding all n values. That is the O(n) space.
- Predict the rows for `trace_two_sum([2, 7, 11, 15], 9)` before running it (one store, then "found at index 0").

### Where it goes wrong

1. **Storing before asking.** The item then finds itself: `two_sum([3, 2, 4], 6)` gives `[0, 0]` (the order lesson of [From Idea to Code](01_Start_Here.ipynb#s01)).
2. **A list as a key.** `groups[counts]` raises `TypeError: unhashable type: 'list'`. Use `tuple(counts)` or `"".join(sorted(w))`.
3. **A fingerprint that forgets something.** `frozenset(w)` drops the counts (`"aab"` and `"abb"` collide), and `ord(ch) - ord("a")` sends capital letters into other letters' slots. The key must be equal *exactly* when the items belong together.
4. **First or last index?** Contains Duplicate II needs the *last* index (overwrite): keeping the first answers `False` for `[1, 0, 1, 1]`, k = 1. "Longest" questions need the *first* (store only when absent).
5. **Walking every member of a run.** In Longest Consecutive, counting up from every value costs 499,500 steps on `range(1000)`. Start only where `x - 1` is missing, and loop over the set (copies in the list would re-walk the run).
6. **Swapping with a computed index.** `nums[i], nums[nums[i] - 1] = nums[nums[i] - 1], nums[i]` builds the right side first and then assigns the targets left to right, so `nums[nums[i] - 1]` is computed with the *new* `nums[i]`: on `[2, 1]` it never ends. Compute `home = nums[i] - 1` first ([Python Toolkit](01_Start_Here.ipynb#s02)).
7. **`int(x / w)` for buckets.** It rounds toward zero, so −3 and 3 share bucket 0 when `w = 5`. Floor division `x // w` keeps negatives in their own buckets.
8. **Voting without a guarantee.** `[1, 2, 3]` has no majority, yet Boyer-Moore returns 3. Unless a majority is promised, count the candidate in a second pass.

### Edge cases to say out loud

Empty input · one item · no pair or no group mate · equal values (`[3, 3]`) · negatives and zero · the empty string as a word · same letters, different counts.

In [ ]:
assert two_sum([], 5) == []
assert two_sum([1, 2], 7) == []                     # no pair
assert two_sum([3, 3], 6) == [0, 1]                 # equal values, two different indices
assert two_sum([3, 2, 4], 6) == [1, 2]              # 3 must not pair with itself
assert two_sum([-1, -2, -3, -4, -5], -8) == [2, 4]  # negatives are just values
assert group_anagrams([]) == []
assert group_anagrams([""]) == [[""]]               # the empty word has a key too (26 zeros)
assert group_anagrams(["ab", "ba", "abc"]) == [["ab", "ba"], ["abc"]]
assert group_anagrams(["aab", "abb"]) == [["aab"], ["abb"]]   # same letters, different counts
print("edge cases pass")

**Try it**
- Add `assert two_sum([0, 4, 3, 0], 0) == [0, 3]`: here `need == v`, and the first 0 is stored before the second one asks.
- Predict, then add: `assert two_sum([5], 10) == []` (one item cannot be its own partner).
- Add `assert group_anagrams(["a", "a"]) == [["a", "a"]]`: equal words share a key, so they land in one group.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Another question, another key** | read the key, the value and when to write off the table "From the question to the code" | 219, 36 |
| **Neighbours by value** | build a set; count up only from run starts (`x - 1` missing) | 128 |
| **Count, then bucket by count** | `Counter`, then `buckets[f]` for f in 1..n, read from high to low | 347 |
| **Vote instead of count** | Boyer-Moore: a candidate and its votes; for "more than n/3", two candidates, then verify | 169, 229 |
| **Encoding** | frame each string as `len#payload`; the decoder jumps, never scans | 271 |
| **Count pairs, meet in the middle** | `pairs += seen[partner]` before `seen[key] += 1`; or count one half's sums, ask with the other's | 1010, 454 |
| **The array is the hash table** (Hard) | value v belongs at index v − 1: swap values home, or mark v by negating `nums[abs(v) - 1]` | 41, 442, 448 |
| **Value buckets, pigeonhole** (Hard) | bucket = `x // width`; close values share a bucket or sit next door | 220, 164 |
| **Counting with order** (Hard) | "how many earlier / later items are smaller": a Fenwick tree over values, or count while merging ([Sorting & Selection](05_Binary_Search_Sorting.ipynb#s23)) | 315, 493, 1649 |

**Neighbours by value.** "Consecutive" is about values, not positions, so sorting looks necessary. A set answers "is x + 1 here?" in O(1) instead. The trick that keeps it O(n): count a run only from its smallest member, the `x` whose `x - 1` is missing.

```text
values:       1 2 3 4 . . . . 100 . . . 200
run starts:   ^                 ^          ^        (x - 1 is missing)
walks:        1->2->3->4        100        200      each value is walked exactly once
```

In [ ]:
def longest_consecutive(nums):
    values = set(nums)                       # STATE: O(1) "is x here?" (built first; drops copies)
    best = 0
    for x in values:
        if x - 1 in values:                  # x sits inside a run: its run start will count it
            continue
        length = 1
        while x + length in values:          # walk up from the run start
            length += 1
        best = max(best, length)             # RECORD
    return best


print(longest_consecutive([100, 4, 200, 1, 3, 2]))            # 4  (1, 2, 3, 4)
print(longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))    # 9  (0 .. 8)

**Try it**
- Add `steps = 0` at the top, `steps += 1` inside the `while`, and print it at the end; run `longest_consecutive(list(range(1000)))`: 999 steps. Now delete the two lines of the `if x - 1 in values` check: 499,500 steps. That `if` is the O(n).
- Loop over `nums` instead of `values` with `[1, 1, 1, 2, 3]`: the answer is still 3, but the run 1-2-3 is walked three times. Many copies of a run start make it quadratic again.
- Predict `longest_consecutive([5, 5, 5])`, `longest_consecutive([])` and `longest_consecutive([-2, -1, 0, 1])` (1, 0 and 4).

**Count, then bucket by count; or vote.** Frequencies are whole numbers from 1 to n, so they can be list indices: `buckets[f]` holds the values seen exactly f times, and reading the buckets from the top gives the most frequent first without sorting. For a value that fills *more than half* the array you need no counts at all: each cancellation removes two different values, so at most one copy of the majority, and the majority outnumbers all the others together, so it survives. (For "more than n/3", keep two candidates and verify them: 229.)

In [ ]:
def top_k_frequent(nums, k):
    count = Counter(nums)                    # STATE: value -> frequency (built first)
    buckets = [[] for _ in range(len(nums) + 1)]   # buckets[f] = values seen exactly f times
    for value, f in count.items():
        buckets[f].append(value)
    out = []
    for f in range(len(nums), 0, -1):        # highest frequency first
        for value in buckets[f]:
            out.append(value)                # RECORD
            if len(out) == k:
                return out
    return out


def majority(nums):                          # assumes some value occurs more than n / 2 times
    candidate, votes = None, 0               # STATE: votes = copies of candidate not yet cancelled
    for v in nums:
        if votes == 0:
            candidate = v                    # everything so far cancelled out: start over with v
        votes += 1 if v == candidate else -1 # STEP: a match adds a vote, anything else cancels one
    return candidate                         # RETURN


print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))   # [1, 2]
print(majority([2, 2, 1, 1, 1, 2, 2]))         # 2

**Try it**
- Size `buckets` with `range(len(nums))` instead of `range(len(nums) + 1)` and run `top_k_frequent([7, 7], 1)`: `IndexError`, because a value can appear n times.
- Run `majority([1, 2, 3])`: it returns 3, yet nothing is a majority. Without the guarantee, count the candidate in a second pass.
- Print `v, candidate, votes` after each step for `[2, 2, 1, 1, 1, 2, 2]`: the votes drop to 0 after the 4th and the 6th item; 1 takes over at the 5th, and 2 takes it back at the 7th.

**Encoding.** Joining with a delimiter breaks as soon as a string contains the delimiter. Writing each string's length first tells the decoder how far to jump, so it never looks inside a payload; the `#` only marks where the length's digits end.

In [ ]:
def encode(strs):
    return "".join(f"{len(s)}#{s}" for s in strs)      # frame = length, '#', payload


def decode(data):
    out, i = [], 0                           # STATE: i = start of the next frame
    while i < len(data):
        j = data.index("#", i)               # the length's digits end at the first '#' from i
        size = int(data[i:j])
        out.append(data[j + 1:j + 1 + size]) # RECORD: exactly `size` chars, whatever they are
        i = j + 1 + size                     # STEP: jump to the next frame
    return out                               # RETURN


packed = encode(["4#a", "", "hi"])
print(packed)                                # 3#4#a0#2#hi
print(decode(packed))                        # ['4#a', '', 'hi']

**Try it**
- Try the delimiter idea: `",".join(["a,b", "c"]).split(",")` gives `['a', 'b', 'c']`, three strings instead of two.
- Print `i` at the top of the decode loop for `encode(["lint", "code"])`: 0, then 6. The pointer lands only on frame starts, never inside a payload.
- Check that `decode(encode([]))` is `[]` while `decode(encode([""]))` is `['']`: the empty list and the list holding one empty string stay different.

**Count pairs, or meet in the middle.** To *count* pairs instead of finding one, ask exactly as in Two Sum but add up the answers: every earlier partner is one more pair. Songs whose total length is a multiple of 60 pair up by remainder: remainder `r` needs `(60 - r) % 60`. With four lists the brute force is n⁴; counting the n² sums of the first two lists, then asking with the n² sums of the other two, is the same lookup with the map built first.

In [ ]:
def count_pairs_divisible_by_60(times):      # 1010
    seen = Counter()                         # STATE: seen[r] = earlier songs with remainder r
    pairs = 0
    for t in times:
        r = t % 60
        pairs += seen[(60 - r) % 60]         # RECORD: every earlier partner makes one more pair
        seen[r] += 1                         # STEP: join the past AFTER asking
    return pairs


def four_sum_count(a, b, c, d):              # 454: build one half, ask with the other
    sums = Counter(x + y for x in a for y in b)       # STATE: a sum from a and b -> how many pairs give it
    return sum(sums[-(x + y)] for x in c for y in d)  # RECORD: each pair from c and d asks for its negation


print(count_pairs_divisible_by_60([30, 20, 150, 100, 40]), count_pairs_divisible_by_60([60, 60, 60]))   # 3 3
print(four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))                                               # 2

**Try it**
- Drop the outer `% 60` (look up `seen[60 - r]`) and run `count_pairs_divisible_by_60([60, 60, 60])`: 0 instead of 3. Remainder 0 needs a partner with remainder 0, not 60.
- Swap the two lines in the loop: `[60, 60, 60]` gives 6 and `[30, 20, 150, 100, 40]` gives 5. Every song whose remainder is its own partner (0 or 30) now pairs with itself.
- Print `Counter(x + y for x in [1, 2] for y in [-2, -1])`: three keys for four pairs, because two pairs share the sum 0. The dict holds at most n² sums, so 454 costs O(n²) time and space instead of O(n⁴).

#### Stretch (Hard)

These three are rarer in interviews. Each still turns a search into a lookup, with a twist.

**The array is the hash table.** First Missing Positive allows only O(1) extra space, so no set. But the answer is in 1..n+1 (n values can cover at most 1..n), and there are exactly n slots, so let slot `v - 1` say "v is present": swap every value 1..n into its home slot, then the first slot without its own value names the answer. A cousin for "find the duplicates / the missing numbers in 1..n" (442, 448): mark v as present by negating `nums[abs(v) - 1]`; a slot that is already negative means v was seen before.

```text
[3, 4, -1, 1]    3 goes to slot 2, 4 to slot 3, 1 to slot 0; -1 has no home and stays where it lands
[1, -1, 3, 4]    slot 1 should hold 2 but doesn't   ->  answer 2
```

In [ ]:
def first_missing_positive(nums):
    n = len(nums)
    for i in range(n):
        # nums[i] can't move when it is out of range, already home, or a copy of what its home holds
        while 1 <= nums[i] <= n and nums[nums[i] - 1] != nums[i]:   # FIX: send nums[i] home
            home = nums[i] - 1               # compute the slot BEFORE swapping
            nums[i], nums[home] = nums[home], nums[i]
    for i in range(n):
        if nums[i] != i + 1:                 # RECORD: the first slot without its own value
            return i + 1
    return n + 1                             # RETURN: 1..n are all present


print(first_missing_positive([3, 4, -1, 1]))   # 2
print(first_missing_positive([1, 2, 0]))       # 3
print(first_missing_positive([7, 8, 9]))       # 1

**Try it**
- Print `nums` after the first loop for `[3, 4, -1, 1]`: `[1, -1, 3, 4]`. Every value 1..n sits at index value − 1, and the hole at index 1 names the answer 2.
- Change the `while` to `if` and rerun `[3, 4, -1, 1]`: 1 instead of 2. The 1 that was swapped into slot 1 never got sent home.
- Count the swaps for `[2, 3, 4, 5, 1]`: 4. Each swap puts one value home for good, so there are at most n swaps in total, even with a loop inside a loop.

**Value buckets.** Contains Duplicate III wants two indices at most k apart whose values are at most t apart. The index part is a window of the last k items; the value part is hashed by cutting the number line into buckets of width t + 1. Two values in one bucket are at most t apart (an instant yes); a close value in another bucket can only be in the bucket next door. Maximum Gap (164) uses the same pigeonhole idea: buckets narrower than the average gap cannot hide the largest gap inside them, so each bucket's min and max are all you keep.

```text
t = 3, width 4:   | 0 1 2 3 | 4 5 6 7 | 8 9 10 11 |
                    bucket 0  bucket 1   bucket 2
x = 5 is in bucket 1: a partner within 3 is in bucket 1 (sure hit) or in bucket 0 / 2 (check the one value there)
```

In [ ]:
def nearby_almost_duplicate(nums, k, t):     # i, j at most k apart with |nums[i] - nums[j]| <= t?
    width = t + 1                            # same bucket  =>  values differ by at most t
    bucket = {}                              # STATE: bucket id -> the one value from the last k indices
                                             # (never two: a second value in a bucket returns True first)
    for i, x in enumerate(nums):
        b = x // width                       # floor division: right for negatives too
        if b in bucket:
            return True                      # RECORD: same bucket
        for nb in (b - 1, b + 1):            # a close value can only live next door
            if nb in bucket and abs(bucket[nb] - x) <= t:
                return True                  # RECORD: a close neighbour
        bucket[b] = x                        # STEP
        if i >= k:
            del bucket[nums[i - k] // width] # FIX: index i - k leaves the window
    return False                             # RETURN


print(nearby_almost_duplicate([1, 2, 3, 1], 3, 0))          # True
print(nearby_almost_duplicate([1, 5, 9, 1, 5, 9], 2, 3))    # False

**Try it**
- Change both `x // width` and `nums[i - k] // width` to `int(... / width)` and run `nearby_almost_duplicate([-3, 3], 2, 4)`: `True`, although |−3 − 3| = 6 > 4. `int()` turns −0.6 into 0, so −3 and 3 share bucket 0.
- Print `bucket` at the end of each step for `[1, 5, 9, 1, 5, 9], 2, 3`: it never holds more than k = 2 values, one per bucket.
- Widen the buckets to `width = t + 2` and run `nearby_almost_duplicate([0, 4], 1, 3)`: `True`, although |0 − 4| = 4 > 3. Width t + 1 is the widest that keeps "same bucket ⇒ at most t apart", and unlike `t` it also works for t = 0.

**Counting with order.** "How many inserted values are smaller than x?" A Counter can't answer that fast; a **Fenwick tree** can. It is a Counter over the values 1..V whose prefix counts cost O(log V): `tree[i]` counts the block of values `(i - lowbit(i), i]`, where `lowbit(i) = i & -i`, so any prefix is a sum of O(log V) blocks. Create Sorted Array (1649) prices each insert with two prefix counts; Count of Smaller Numbers After Self (315) fills the same tree from the right. Merge sort counts such pairs too: see *count while merging* in [Sorting & Selection](05_Binary_Search_Sorting.ipynb#s23), which also cracks Reverse Pairs (493).

```text
values 1..8:   tree[8] counts 1..8   tree[4] counts 1..4   tree[6] counts 5..6   tree[7] counts 7
"how many values <= 7" = tree[7] + tree[6] + tree[4]          (7 -> 6 -> 4 -> 0: drop the lowest set bit)
```

In [ ]:
def create_sorted_array_cost(instructions):  # 1649: inserting x costs min(# smaller, # larger) so far
    size = max(instructions)
    tree = [0] * (size + 1)                  # STATE: Fenwick tree over values 1..size (index 0 unused)

    def add(v):                              # one more copy of v
        while v <= size:
            tree[v] += 1
            v += v & -v                      # the next block that also covers v

    def count_upto(v):                       # how many inserted values are <= v
        total = 0
        while v > 0:
            total += tree[v]
            v -= v & -v                      # drop the lowest set bit: the block to the left
        return total

    cost = 0
    for inserted, x in enumerate(instructions):          # `inserted` values are in the tree
        cost += min(count_upto(x - 1), inserted - count_upto(x))   # RECORD: smaller vs larger
        add(x)                               # STEP: x joins AFTER it is priced
    return cost                              # RETURN (LeetCode wants it modulo 10**9 + 7)


print(create_sorted_array_cost([1, 5, 6, 2]))                  # 1  (2 has one smaller, two larger)
print(create_sorted_array_cost([1, 3, 3, 3, 2, 4, 2, 1, 2]))   # 4

**Try it**
- Use `count_upto(x)` for the smaller side and rerun the second example: 8 instead of 4. Copies of x were counted as smaller.
- Print `tree` before the `return` for `create_sorted_array_cost(list(range(1, 9)))`: `[0, 1, 2, 1, 4, 1, 2, 1, 8]`. Each entry counts a block that ends at its index, `i & -i` values long.
- For 315, number the distinct values 1..m (`rank`), walk `nums` from the right, record `count_upto(rank[x] - 1)` and then `add(rank[x])`: `[5, 2, 6, 1]` gives `[2, 1, 1, 0]`.

### Say it in the interview

> "The brute force fixes one number and scans the rest for its partner: O(n²). But that scan is a search for one known value, target − v, and a hash map answers 'is this value here, and where?' in O(1) on average. So I'll walk once with a dict from value to index: for each number I look up its partner, and only then store the number, so it can't pair with itself. O(n) time on average, O(n) space."

Then say what the map means ("value to the index of an earlier copy") and point at the two lines whose order matters: the lookup, then the store. For grouping problems, name the fingerprint and why equal fingerprints mean "same group". Follow-ups to have ready:

- *Sorted input, or O(1) extra space?* [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05). To keep the original indices, sort `(value, index)` pairs: O(n log n).
- *Count all pairs instead of finding one?* `pairs += seen[need]` before `seen[v] += 1`.
- *Worst case?* If every key collides, one lookup degrades to O(n); with Python's hashing of numbers and strings that is rare.
- *Group Anagrams cost?* O(n·k) with the count key, O(n·k log k) with the sorted key, for n words of length k.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Contains Duplicate II | `arrays_hashing/contains_duplicate_ii.py` | store each value's last index; only the nearest earlier copy can be within k |
| Contains Duplicate III | `arrays_hashing/contains_duplicate_iii.py` | buckets of width t + 1 over the last k values: same bucket is a hit, else check both neighbours |
| Count of Smaller Numbers After Self | `arrays_hashing/count_of_smaller_numbers_after_self.py` | count, from the right, the smaller values already seen: a Fenwick tree over ranks, or count while merging |
| Create Sorted Array through Instructions | `arrays_hashing/create_sorted_array_through_instructions.py` | Fenwick tree over values: cost of x = min(prefix(x − 1), inserted − prefix(x)) |
| Encode and Decode Strings | `arrays_hashing/encode_and_decode_strings.py` | frame each string as `len#payload`; the decoder jumps over payloads, so any character is safe |
| First Missing Positive | `arrays_hashing/first_missing_positive.py` | the answer is in 1..n+1: swap each v into slot v − 1, then find the first wrong slot |
| Group Anagrams | `arrays_hashing/group_anagrams.py` · `practice/simple/02_group_anagrams.py` | fingerprint = 26 letter counts as a tuple (or the sorted word); bucket words by it |
| Longest Consecutive Sequence | `arrays_hashing/longest_consecutive_sequence.py` · `practice/simple/04_longest_consecutive_sequence.py` | set of values; count up only from run starts (x − 1 missing), so each value is walked once |
| Majority Element | `arrays_hashing/majority_element.py` | Boyer-Moore: different values cancel in pairs and the majority survives |
| Maximum Gap | `arrays_hashing/maximum_gap.py` | buckets narrower than the average gap: the max gap runs between neighbouring non-empty buckets |
| Reverse Pairs | `arrays_hashing/reverse_pairs.py` | merge sort: on two sorted halves, count x > 2r with a forward-only pointer, then merge |
| Top K Frequent Elements | `arrays_hashing/top_k_frequent_elements.py` | count, then bucket values by frequency (1..n) and read the buckets from high to low |
| Two Sum | `arrays_hashing/two_sum.py` · `practice/simple/01_two_sum.py` | dict value → index; look up target − v before storing v |
| Valid Sudoku | `arrays_hashing/valid_sudoku.py` | one set of (unit, digit) pairs, or a set per row, column and box; box id = (r // 3, c // 3) |

### Self-check

1. When do you build the whole map first instead of asking, then storing, item by item?
<details><summary>Answer</summary>When the question is about the whole collection (frequencies, the k most frequent, runs of consecutive values): an item's answer depends on items that come after it, so the map must be complete before you read it. Ask-then-store fits questions that pair an item with an <em>earlier</em> one, like Two Sum.</details>

2. Longest Consecutive Sequence has a `while` inside a `for`. Why is it still O(n)?
<details><summary>Answer</summary>The <code>while</code> runs only from run starts (values whose <code>x - 1</code> is missing), and each run is walked once from its start. Every value belongs to exactly one run, so all the walks together take at most n steps.</details>

3. When does Boyer-Moore voting give a wrong answer, and how do you protect against it?
<details><summary>Answer</summary>When no value occurs more than n / 2 times it still returns some candidate (<code>[1, 2, 3]</code> gives 3). If a majority is not guaranteed, make a second pass that counts the candidate and checks <code>count > n // 2</code>.</details>

4. In First Missing Positive, why is it safe to leave values ≤ 0 and values > n wherever they land?
<details><summary>Answer</summary>The answer is the smallest of 1..n+1 that is missing. Values outside 1..n can never fill one of the slots for 1..n, so they cannot change which of those numbers is missing first; they just occupy slots that will be reported as holes.</details>

<a id="s04"></a>

## Prefix Sums

> Write down the running total at every position once; then the sum of any stretch is the difference of two totals, like the distance between two towns read off a car's odometer. Questions about subarrays become questions about *pairs of totals*, and a list or a dict answers those in O(1).

**Reach for it when** a problem asks about sums (or products, counts, parities) of *contiguous* stretches: many range-sum queries, "count / longest subarrays with sum k" (especially with negatives, where a [Sliding Window](03_Two_Pointers_Sliding_Window_Strings.ipynb#s06) fails), "product of everything except me", "equal number of 0s and 1s", "add v to every element in [l, r]" over and over, the sum of a rectangle in a grid, or a random pick weighted by sizes.

**In this repo:** `arrays_hashing/subarray_sum_equals_k.py`, `arrays_hashing/product_of_array_except_self.py`, `arrays_hashing/max_consecutive_ones.py` · bank: `practice/simple/05_subarray_sum_equals_k.py`, `practice/simple/03_product_of_array_except_self.py` · basics: `practice/simple/basics/bits/03_xor_tricks_single_number_missing_number.py` (why xor can be "subtracted") · the same idea elsewhere: `intervals/my_calendar_iii.py` (difference array, [Intervals & Sweep Line](07_Heaps_Intervals_Greedy.ipynb#s14)), `bit_manipulation/find_longest_awesome_substring.py` (prefix parity mask, [Math, Bits & Geometry](10_Matrices_Math_Bits.ipynb#s22)), `greedy/super_washing_machines.py` (prefix-sum flow), `greedy/maximum_subarray.py` (running sum, [Greedy](07_Heaps_Intervals_Greedy.ipynb#s15)), `design/range_sum_query_2d_mutable.py` (2D range sums with updates, [Design](11_Design.ipynb#s24))

### The picture

Think of the totals as **fence posts between the items**: post `P[k]` stands just before item k and holds the sum of everything to its left.

```text
posts:     P0    P1    P2    P3    P4    P5
            |  3  |  1  | -2  |  4  |  1  |
totals:     0     3     4     2     6     7          P[k] = sum of the first k items

the items between two posts sum to (right post) - (left post):
items 1..3 = 1 + (-2) + 4 = 3 = P4 - P1 = 6 - 3
```

Items `l..r` sit between post `l` and post `r + 1`; that is where the famous `+ 1` comes from.

Why it is fast: the brute force adds the same items again for every (start, end) pair. Prefix sums do the adding once, in O(n); after that every range costs one subtraction.

The second half of the trick turns "for every start" into a dict lookup. To count stretches that sum to k, walk the posts from left to right and ask at each one: *how many earlier posts hold my total minus k?*

```text
k = 3       totals:  0   3   4   2   6   7
                     P0  P1  P2  P3  P4  P5

at P1 (3): look for 0   -> P0      items 0..0  = [3]
at P4 (6): look for 3   -> P1      items 1..3  = [1, -2, 4]
at P5 (7): look for 4   -> P2      items 2..4  = [-2, 4, 1]          3 stretches in one pass
```

The lookup never needs the totals to be in order, so negative numbers are fine. A sliding window would need the sum to grow every time the window grows.

### From idea to code

**The idea in one sentence:** *carry the running total as you walk; a stretch ending here sums to `total - (an earlier total)`, so a question about stretches becomes a question about one earlier total, which a list (by position) or a dict (by value) answers instantly.*

**It is Two Sum on the running totals.** Subarray Sum Equals K looks up `total - k` where [Two Sum](#s03) looks up `target - v`, and it *counts* instead of returning. The seed `{0: 1}` is post P0, the empty prefix, already in the notebook before the first item.

| Decision | Prefix-sum answer |
|---|---|
| **State**: what must I remember? | the running `total` (or the whole table `P` when the queries come later); for counting, `posts[t]` = how many earlier posts hold total `t` |
| **Definition**: what exactly does each variable mean? | `P[k] = nums[0] + ... + nums[k-1]` (so `P[0] = 0`), and then `sum(nums[l..r]) = P[r + 1] - P[l]` |
| **Invariant**: what is true at the end of every step? | after the query and before the store: `total == P[j + 1]`, and `posts` holds exactly the earlier posts `P[0..j]` |
| **Step**: how does one item change the state? | `total += x`; then, after the query, add `total` to `posts` |
| **Record**: when is the answer updated? | `found += posts.get(total - k, 0)`, *before* storing the current post |
| **Init**: starting values | `total = 0` and `posts = {0: 1}`: the empty prefix is a real post, needed by stretches that start at index 0 |
| **Return**: what comes back, and for "not found"? | `found` (0 when nothing matched), or `P[r + 1] - P[l]` for each query |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "the total of the first k items" | `P[k]`; the whole table at once: `P = [0] + list(accumulate(nums))` |
| "the sum of items l..r, both included" | `P[r + 1] - P[l]` |
| "the running total so far" | `total += x` |
| "a stretch ending here sums to k" | `total - k in posts` |
| "how many such stretches" | `posts.get(total - k, 0)` |
| "the empty prefix counts" | `posts = {0: 1}` |
| "remember this post" | `posts[total] = posts.get(total, 0) + 1`, after the query |

**From the question to the code.** Most prefix-sum questions are this one loop with four knobs: what each item adds to the total, what you look up, what the dict stores, and its seed.

| Question | Each item adds | Look up | Dict stores | Seed |
|---|---|---|---|---|
| count stretches with sum k (560) | `x` | `total - k` | posts per total | `{0: 1}` |
| longest stretch with sum k (325) | `x` | `total - k` | first index per total | `{0: -1}` |
| shortest stretch with sum k | `x` | `total - k` | last index per total | `{0: -1}` |
| as many 0s as 1s (525) | `+1` or `-1` | the same total | first index | `{0: -1}` |
| count sums divisible by k (974) | `x`, keep `% k` | the same remainder | posts per remainder | `{0: 1}` |
| a multiple of k, length ≥ 2 (523) | `x`, keep `% k` | the same remainder; a hit needs `j - first >= 2` | first index | `{0: -1}` |
| exactly k odd numbers (1248) | `x % 2` | `total - k` | posts per total | `{0: 1}` |
| every vowel an even number of times (1371) | flip the vowel's bit | the same mask | first index | `{0: -1}` |
| at most one digit odd (1542) | flip the digit's bit | the same mask, or one bit flipped | first index | `{0: -1}` |

When the dict stores indices, name each post by the item it follows: after item `j` the post is called `j`, and the empty prefix is called `−1`. The stretch after post `i` up to item `j` then has length `j − i`.

Two templates: a table for range queries, and the running total plus dict for counting. The second is [From Idea to Code](01_Start_Here.ipynb#s01)'s `subarray_sum` with a plain dict and the picture's names. Its RECORD comes before the last STEP, so the current post can never pair with itself.

In [ ]:
def prefix_table(nums):
    P = [0] * (len(nums) + 1)                # STATE + INIT: P[k] = sum of the first k items; P[0] = 0
    for k, x in enumerate(nums):
        P[k + 1] = P[k] + x                  # STEP: one more item on the running total
    return P


def range_sum(P, l, r):                      # RETURN: sum of nums[l..r], both ends included
    return P[r + 1] - P[l]


def count_subarrays_with_sum(nums, k):
    posts = {0: 1}                           # STATE + INIT: total -> how many posts hold it; the empty prefix
    total = found = 0
    for x in nums:
        total += x                           # STEP: total = sum of everything up to and including x
        found += posts.get(total - k, 0)     # RECORD: each earlier post holding total - k starts a k-sum
        posts[total] = posts.get(total, 0) + 1   # STEP: this post joins AFTER the query
    return found                             # RETURN


P = prefix_table([3, 1, -2, 4, 1])
print(P, range_sum(P, 1, 3))                           # [0, 3, 4, 2, 6, 7] 3
print(count_subarrays_with_sum([3, 1, -2, 4, 1], 3))   # 3

**Try it**
- Keep a *set* of totals instead of counts (`posts = {0}`, `found += (total - k) in posts`, `posts.add(total)`) and run `count_subarrays_with_sum([0, 0, 0], 0)`: 3 instead of 6. Equal totals are different starts, and each one is a separate answer.
- Predict `count_subarrays_with_sum([1, 2, 3], 3)` and name each stretch's earlier post (2: `[1, 2]` pairs with P0, `[3]` with P2).
- Predict `range_sum(P, 0, 4)` and `range_sum(P, 2, 2)` before running them (7, the whole array, and −2, a single item).

### Watch it work

Each line is one item; `posts` is shown *before* the current post is stored.

In [ ]:
def trace_count(nums, k):
    posts, total, found = {0: 1}, 0, 0
    for j, x in enumerate(nums):
        total += x
        hits = posts.get(total - k, 0)
        found += hits
        print(f"j={j}  x={x:>2}  total={total:>2}  look for {total - k:>2}: {hits} hit(s)  found={found}   posts={posts}")
        posts[total] = posts.get(total, 0) + 1
    return found


print(trace_count([3, 1, -2, 4, 1], 3))   # 3

**Try it**
- Run `trace_count([0, 0, 0], 0)`: the hits are 1, 2 and 3, for a total of 6. Every pair of equal totals is one zero-sum stretch.
- Run `trace_count([1, -1, 1, -1], 0)` and match each hit to the earlier post it pairs with (4 stretches in all).
- Predict the answer before running `trace_count([3, 1, -2, 4, 1], 4)` (3: `[3, 1]`, `[4]` and `[1, -2, 4, 1]`).

### Where it goes wrong

1. **The empty prefix, and the order of the two dict lines.** Both are [From Idea to Code](01_Start_Here.ipynb#s01) lessons: without `{0: 1}`, `[3]` with k = 3 gives 0; storing before the query makes `[1]` with k = 0 give 1.
2. **Off by one in ranges.** `sum(nums[l..r]) = P[r + 1] - P[l]`. `P[r] - P[l]` leaves out item r; `P[r + 1] - P[l - 1]` adds item l − 1 (and at l = 0, `P[-1]` silently reads the *last* total).
3. **First, last, or how many?** Counting stores how many posts hold each total. *Longest* keeps the **first** index (store only when absent, in its own `if`, not an `else`); *shortest* keeps the **last** (always overwrite). Overwriting in a "longest" question turns Contiguous Array on `[0, 1, 0, 1]` into 2 instead of 4.
4. **Sliding window with negatives.** A window that shrinks while its sum is above k counts 2 on `[1, -1, 1]` with k = 1; the answer is 3 (`[1]`, `[1]` and `[1, -1, 1]`). Dropping an item can raise the sum, so the window cannot tell when to stop.
5. **Including yourself in a left-right pass.** In Product Except Self, multiply `nums[i]` into the running product *after* stamping `out[i]`, in both passes; otherwise `[1, 2, 3, 4]` gives `[24, 48, 72, 96]`. And skip division: it fails on zeros.
6. **Ragged 2D borders.** Give the 2D table an extra zero row and column, `(R + 1) × (C + 1)`; then no `if r > 0` cases are needed and the build is `cell + up + left - up_left`.
7. **Difference array: one slot short, or the wrong end.** `diff[r + 1] -= v` needs `n + 1` slots, or an update that ends at the last index raises `IndexError`. For half-open ranges `[start, end)` the −v goes at `end` itself: treating `end` as included makes `[10, 20)` and `[20, 30)` overlap at 20.

### Edge cases to say out loud

Empty input · one item equal to k · `k = 0` · all zeros (many overlapping answers) · negatives · the whole array is the answer · no answer at all.

In [ ]:
assert count_subarrays_with_sum([], 0) == 0
assert count_subarrays_with_sum([3], 3) == 1               # needs the {0: 1} seed
assert count_subarrays_with_sum([1], 0) == 0               # query before store
assert count_subarrays_with_sum([0, 0, 0], 0) == 6         # all 6 stretches
assert count_subarrays_with_sum([1, -1, 1, -1], 0) == 4    # negatives are fine
assert count_subarrays_with_sum([1, 2, 3], 7) == 0         # no answer
assert prefix_table([]) == [0]                             # one post, no items
assert range_sum(prefix_table([5]), 0, 0) == 5
print("edge cases pass")

**Try it**
- Predict `count_subarrays_with_sum([0], 0)` before running it (1: the single 0 pairs with the empty prefix).
- Predict, then add: `assert count_subarrays_with_sum([1, 2, 3], 6) == 1` (only the whole array).
- A negative k works too: predict `count_subarrays_with_sum([-1, -1, 1], -2)` (1: the first two items).

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Range sum queries** | build `P` once; each query is `P[r + 1] - P[l]` | 303 |
| **The dict recipes** | change what an item adds, what you look up and what the dict stores (the table above) | 560, 325, 525, 974, 523, 1248, 1371, 1542 |
| **Left-right passes** | carry a running value from each side. 238 stamps before multiplying (never include yourself); 42 includes the bar in its max, so `min(L, R) - h` is never negative | 238, 42 |
| **Running value that resets** | carry `run`; a wall resets it to 0; record the max inside the run | 485 |
| **2D prefix sums** | `(R + 1) × (C + 1)` table; a rectangle is four corners, inclusion–exclusion | 304, 1074 |
| **Difference array** | range update: `diff[l] += v`, `diff[r + 1] -= v` (half-open: at `end`); one prefix sum rebuilds the values | 370, 1109, 1094, 732 |
| **Prefix sums + binary search** | positive weights make the totals increasing, so `bisect` finds whose ticket was drawn | 528 |

**Longest instead of count; remainders instead of totals.** For the longest stretch, keep the *first* index where each total appeared: the earliest start gives the longest stretch. Contiguous Array (525) is this with every 0 turned into −1 and k = 0. For "divisible by k", two posts with the same remainder are a multiple of k apart.

In [ ]:
def longest_with_sum(nums, k):               # 325; 525 is this on +1 / -1 with k = 0
    first = {0: -1}                          # STATE + INIT: total -> FIRST post (item index) holding it
    total = best = 0
    for j, x in enumerate(nums):
        total += x                           # STEP
        if total - k in first:               # RECORD: the earliest post that leaves exactly k
            best = max(best, j - first[total - k])
        if total not in first:               # STEP: store the first time only, and NOT as an else
            first[total] = j
    return best


def count_divisible(nums, k):                # 974: stretches whose sum is divisible by k
    posts = {0: 1}                           # STATE + INIT: remainder -> how many posts hold it
    total = found = 0
    for x in nums:
        total = (total + x) % k              # STEP: Python's % stays in 0..k-1 for k > 0, even for negatives
        found += posts.get(total, 0)         # RECORD: equal remainders, so the difference is a multiple of k
        posts[total] = posts.get(total, 0) + 1
    return found


print(longest_with_sum([1, -1, 5, -2, 3], 3))                                        # 4  ([1, -1, 5, -2])
print(longest_with_sum([1 if b else -1 for b in [0, 1, 1, 0, 1, 1, 1, 0]], 0))       # 4  (525: [0, 1, 1, 0])
print(count_divisible([4, 5, 0, -2, -3, 1], 5))                                      # 7

**Try it**
- Turn the second `if` into an `else` and run `longest_with_sum([1, -1, 1], 1)`: 1 instead of 3. On the miss at index 1 the `else` overwrote the empty prefix's `-1`, which the whole array needed.
- Always overwrite (`first[total] = j` with no `if`) and run `longest_with_sum([-1, 1, -1, 1], 0)`: 2 instead of 4. That is the rule for *shortest*, not longest.
- Drop `% k` in `count_divisible` and run it on `[5, 5]` with k = 5: 0 instead of 3.

**Left-right passes and running values.** "Everything except me" is (everything to my left) × (everything to my right), and both parts are running products: one pass from the left stamps the left part into `out`, one pass from the right multiplies in the right part. Trapping Rain Water's first solution has the same shape with running *maxima*, each including the bar itself so the water `min(L, R) - h` is never negative; the [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05) section then removes the arrays. A running value can also reset: Max Consecutive Ones carries the length of the current run of 1s, and every 0 is a wall that sends it back to 0.

```text
nums           1    2    3    4
left part      1    1    2    6        product of everything left of i
right part    24   12    4    1        product of everything right of i
out           24   12    8    6        left part * right part
```

In [ ]:
def product_except_self(nums):
    n = len(nums)
    out = [1] * n
    left = 1                                 # STATE: product of nums[0..i-1]
    for i in range(n):
        out[i] = left                        # RECORD: stamp first ...
        left *= nums[i]                      # STEP: ... then include nums[i] for the indices after it
    right = 1                                # STATE: product of nums[i+1..n-1]
    for i in range(n - 1, -1, -1):
        out[i] *= right
        right *= nums[i]
    return out


def max_consecutive_ones(nums):
    run = best = 0                           # STATE: run = length of the run of 1s ending at this item
    for x in nums:
        if x == 1:
            run += 1                         # STEP
            best = max(best, run)            # RECORD inside the run, so a run at the very end counts
        else:
            run = 0                          # STEP: a 0 is a wall, no run crosses it
    return best


print(product_except_self([1, 2, 3, 4]))          # [24, 12, 8, 6]
print(product_except_self([-1, 1, 0, -3, 3]))     # [0, 0, 9, 0, 0]
print(max_consecutive_ones([1, 1, 0, 1, 1, 1]))   # 3

**Try it**
- Swap the two lines inside *both* loops of `product_except_self` (multiply first, then stamp) and run `[1, 2, 3, 4]`: `[24, 48, 72, 96]`. Every index now includes itself.
- Run `product_except_self([0, 0, 2])`: `[0, 0, 0]`, with no special case for zeros (dividing a total product would crash here).
- In `max_consecutive_ones`, move `best = max(best, run)` into the `else` branch (above `run = 0`) and run `[0, 1, 1, 1]`: 0 instead of 3. The last run never meets a 0, so it is never recorded.

**2D prefix sums.** `P[r][c]` is the sum of the rectangle above and to the left of the crossing of post row r and post column c. Any rectangle is the big one, minus the strip above, minus the strip to the left, plus the corner that was subtracted twice.

```text
                 post c1      post c2+1
                    |              |
              A     |      B       |
  post r1   --------+--------------+       rect = P[r2+1][c2+1]     (A + B + C + rect)
              C     |     rect     |            - P[r1][c2+1]       (A + B)
  post r2+1 --------+--------------+            - P[r2+1][c1]       (A + C)
                                                + P[r1][c1]         (A, which was subtracted twice)
```

In [ ]:
def prefix_2d(grid):
    R, C = len(grid), len(grid[0])
    P = [[0] * (C + 1) for _ in range(R + 1)]    # STATE + INIT: P[r][c] = sum of grid[0..r-1][0..c-1]; zero border
    for r in range(R):
        for c in range(C):
            P[r + 1][c + 1] = grid[r][c] + P[r][c + 1] + P[r + 1][c] - P[r][c]   # STEP: cell + up + left - overlap
    return P


def rect_sum(P, r1, c1, r2, c2):             # RETURN: cells r1..r2 x c1..c2, corners included
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]


grid = [[3, 0, 1],
        [5, 6, 3],
        [1, 2, 0]]
P2 = prefix_2d(grid)
print(rect_sum(P2, 1, 1, 2, 2))   # 11  (6 + 3 + 2 + 0)
print(rect_sum(P2, 0, 0, 2, 2))   # 21  (the whole grid)

**Try it**
- Print `P2` row by row: the first row and the first column are all zeros. That border is why neither function needs an `if r > 0` check.
- Drop `- P[r][c]` from the build and rerun: `rect_sum(P2, 0, 0, 2, 2)` becomes 52 instead of 21. The shared corner is added twice at every cell, and the error snowballs.
- Predict `rect_sum(P2, 2, 0, 2, 2)` (the bottom row: 3) and `rect_sum(P2, 1, 0, 1, 0)` (a single cell: 5).

**Difference array.** The reverse trick: to add v to a whole range many times, record only where the change *starts* (`+v` at l) and where it *stops* (`-v` at r + 1). One prefix sum at the end rebuilds every value. Car Pooling (1094) and My Calendar III (732, in [Intervals & Sweep Line](07_Heaps_Intervals_Greedy.ipynb#s14)) use it on a timeline of half-open trips `[start, end)`: +1 at each start, −1 at each end itself, and the running sum is how many trips overlap at that moment.

```text
add +2 on 1..3:    diff   0  +2   0   0  -2   0        running sum ->   0  2  2  2  0
```

In [ ]:
def apply_range_adds(n, updates):            # updates: (l, r, v) = add v to every index in l..r
    diff = [0] * (n + 1)                     # STATE + INIT: one extra slot, so diff[r + 1] exists for r = n - 1
    for l, r, v in updates:
        diff[l] += v                         # STEP: from l on, everything is v higher ...
        diff[r + 1] -= v                     # ... and from r + 1 on, back to normal
    out, running = [], 0
    for i in range(n):
        running += diff[i]                   # the prefix sum of the changes is the value at i
        out.append(running)
    return out                               # RETURN


print(apply_range_adds(5, [(1, 3, 2), (2, 4, 3), (0, 1, -1)]))   # [-1, 1, 5, 5, 3]

**Try it**
- Print `diff` before the rebuild loop: `[-1, 2, 4, 0, -2, -3]`. Each update touched only two slots.
- Make `diff = [0] * n` and run `apply_range_adds(5, [(2, 4, 3)])`: `IndexError`, because the update ends at the last index and `diff[r + 1]` falls off the end.
- Feed two half-open bookings `[10, 20)` and `[20, 30)` as `(10, 20, 1)` and `(20, 30, 1)` with n = 31: the maximum is 2, yet they never overlap. Pass `end - 1` instead (`(10, 19, 1)`, `(20, 29, 1)`) and it is 1.

**Prefix xor / parity mask.** Xor is addition where 1 + 1 = 0, so it tracks *parity*. Give each vowel one bit; the running mask says which vowels have appeared an odd number of times so far. Two equal masks mean every vowel appeared an even number of times in between: the "first index of each total" trick again, with masks as the totals. [Math, Bits & Geometry](10_Matrices_Math_Bits.ipynb#s22) applies it to digits, where one odd digit is allowed (1542).

In [ ]:
def longest_even_vowels(s):                  # longest substring where every vowel count is even (1371)
    bit = {v: 1 << i for i, v in enumerate("aeiou")}
    first = {0: -1}                          # STATE + INIT: mask -> first post holding it; empty prefix at -1
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= bit.get(ch, 0)               # STEP: flip this vowel's parity; other letters change nothing
        if mask in first:
            best = max(best, j - first[mask])   # RECORD: same parities at both ends, even counts between
        else:
            first[mask] = j
    return best


print(longest_even_vowels("eleetminicoworoep"))   # 13
print(longest_even_vowels("bcbcbc"))              # 6

**Try it**
- Print `j, ch, f"{mask:05b}"` at each step for `"abca"`: the a-bit turns on at index 0 and off at index 3, so mask 0 comes back and the whole string (length 4) counts.
- Run `longest_even_vowels("leetcodeisgreat")`: 5 (`"leetc"`, which has two e's).
- Allow one odd vowel, the 1542 rule: also look up `mask ^ b` for every vowel bit `b` (`for b in bit.values(): if mask ^ b in first: best = max(best, j - first[mask ^ b])`). `longest_even_vowels("eae")` goes from 0 to 3.

**Prefix sums + binary search.** To pick index i with probability `w[i] / sum(w)`, give index i a block of `w[i]` tickets on the number line 1..total. The running totals mark where each block ends, and with positive weights they increase, so `bisect` finds the block a random ticket falls in.

```text
w = [1, 3, 6]    P = [1, 4, 10]
tickets:   1 | 2 3 4 | 5 6 7 8 9 10
index:     0 |   1   |      2
```

In [ ]:
class WeightedPicker:                        # 528: pick i with probability w[i] / sum(w)
    def __init__(self, w):
        self.P = list(accumulate(w))         # STATE: index i owns tickets P[i-1]+1 .. P[i]

    def pick(self):
        x = random.randint(1, self.P[-1])    # draw a ticket 1..total
        return bisect.bisect_left(self.P, x) # RETURN: the first index whose running total reaches x


random.seed(0)
picker = WeightedPicker([1, 3, 6])
print(sorted(Counter(picker.pick() for _ in range(10000)).items()))   # [(0, 1011), (1, 3009), (2, 5980)]  (about 1 : 3 : 6)

**Try it**
- Use `bisect_right` and see where each ticket goes for `w = [1, 3]`: `[bisect.bisect_right([1, 4], x) for x in range(1, 5)]` is `[1, 1, 1, 2]`, so ticket 4 lands on index 2, which does not exist; `bisect_left` gives `[0, 1, 1, 1]`.
- Give some items weight 0: `WeightedPicker([0, 5, 0])` only ever returns 1, because indices 0 and 2 own no tickets.
- Draw from `random.randint(0, self.P[-1])` (ticket 0 included): index 0 now owns 2 tickets out of 11 instead of 1 out of 10.

### Say it in the interview

> "Brute force tries every start and end: O(n²) even with a running sum. A subarray sum is a difference of two prefix sums, so this is Two Sum on the running totals: for each end I need the number of earlier prefix sums equal to the current one minus k. A dict of prefix-sum counts, seeded with {0: 1} for the empty prefix, answers that in O(1) on average. One pass, O(n) time and O(n) space, and it works with negatives, unlike a sliding window."

Then point at the seed ("the empty prefix, for subarrays that start at index 0") and at the order of the two dict lines: query first, then store. Follow-ups to have ready:

- *Values change between queries?* A Fenwick tree keeps prefix sums with O(log n) updates ([Arrays & Hashing](#s03) builds one over values, [Design](11_Design.ipynb#s24) one in 2D).
- *Longest instead of how many? Return the subarray itself?* Store the first index of each total; the stretch is `first[total - k] + 1 .. j`.
- *Count submatrices with sum k (1074)?* Fix a pair of rows, collapse the columns between them into one array, and run this dict: O(R²·C).
- *Maximum subarray?* The largest `P[j] − (smallest earlier P)`: the buy-and-sell loop of [From Idea to Code](01_Start_Here.ipynb#s01) run on the totals, which [Greedy](07_Heaps_Intervals_Greedy.ipynb#s15) writes as Kadane.
- *Downward paths in a tree that sum to k (437)?* The same dict over root-to-node totals, with `posts[total] -= 1` as the DFS leaves a node.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Max Consecutive Ones | `arrays_hashing/max_consecutive_ones.py` | carry the current run length; a 0 resets it; record the max inside the run so a final run counts |
| Product of Array Except Self | `arrays_hashing/product_of_array_except_self.py` · `practice/simple/03_product_of_array_except_self.py` | out[i] = product left of i × product right of i: one pass each way, stamp before multiplying in |
| Subarray Sum Equals K | `arrays_hashing/subarray_sum_equals_k.py` · `practice/simple/05_subarray_sum_equals_k.py` | Two Sum on the running totals: count earlier totals equal to total − k; seed {0: 1}; query before storing |

### Self-check

1. In what sense is Subarray Sum Equals K "Two Sum on the running totals"?
<details><summary>Answer</summary>A stretch ending at item j sums to k exactly when an earlier post holds <code>total - k</code>, just as Two Sum's partner is <code>target - v</code>. The dict maps a total to how many earlier posts hold it (instead of value to index), and the answer adds up the hits instead of returning the first one. The seed <code>{0: 1}</code> is the empty prefix, already in the notebook.</details>

2. For "longest subarray with sum k", why do you store only the *first* index of each total?
<details><summary>Answer</summary>The stretch runs from just after the earlier post to the current item, so the earlier that post, the longer the stretch. Overwriting with later indices would only ever shorten the answers (that is the rule for <em>shortest</em>).</details>

3. Why does a sliding window fail on "count subarrays with sum k" with negative numbers, while prefix sums + a dict work?
<details><summary>Answer</summary>The window decides to shrink because the sum is too big, which only makes sense if dropping an item lowers the sum. A negative item breaks that, so there is no safe moment to shrink (<code>[1, -1, 1]</code>, k = 1: the window counts 2 of the 3). The dict approach never shrinks anything: it looks up an exact earlier total, which works for any numbers.</details>

4. In a difference array, what does each update touch, and why is `diff` one slot longer than the array?
<details><summary>Answer</summary>Each update touches two slots: <code>+v</code> where the range starts and <code>-v</code> just after it ends, at <code>r + 1</code>. When the range ends at the last index, <code>r + 1 = n</code>, so <code>diff</code> needs n + 1 slots. A prefix sum over <code>diff</code> then rebuilds every value.</details>